In [1]:
# Neural Identifier Training - 2-DOF Planar Manipulator
# Methods: EKF, UKF, Particle Filter

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# 1) True nonlinear system (2-DOF Robot Arm)
# ============================================================
def plant_dynamics(state, u):
    """
    Dynamics of a 2-link planar robot arm.
    state = [q1, q2, dq1, dq2] (Angles and Velocities)
    u     = [tau1, tau2]       (Torques)
    """
    # Robot Parameters
    m1, m2 = 1.0, 1.0  # Mass (kg)
    l1, l2 = 1.0, 1.0  # Lengths (m)
    g = 9.81
    
    q1, q2, dq1, dq2 = state
    tau1, tau2 = u

    # --- Mass Matrix M(q) ---
    c2 = np.cos(q2)
    s2 = np.sin(q2)
    
    M11 = (m1 + m2) * l1**2 + m2 * l2**2 + 2 * m2 * l1 * l2 * c2
    M12 = m2 * l2**2 + m2 * l1 * l2 * c2
    M21 = M12
    M22 = m2 * l2**2
    M = np.array([[M11, M12], [M21, M22]])

    # --- Coriolis/Centrifugal Matrix C(q, dq) ---
    h = -m2 * l1 * l2 * s2
    C11 = h * dq2
    C12 = h * (dq1 + dq2)
    C21 = -h * dq1
    C22 = 0.0
    C = np.array([[C11, C12], [C21, C22]])

    # --- Gravity Vector G(q) ---
    s1 = np.sin(q1)
    s12 = np.sin(q1 + q2)
    G1 = (m1 + m2) * g * l1 * s1 + m2 * g * l2 * s12
    G2 = m2 * g * l2 * s12
    G = np.array([G1, G2])

    # --- Equation of Motion: M*ddq + C*dq + G = tau ---
    # ddq = inv(M) * (tau - C*dq - G)
    
    damping = 0.5 * np.array([dq1, dq2]) # Add some friction
    torque_vector = np.array([tau1, tau2])
    
    rhs = torque_vector - (C @ np.array([dq1, dq2])) - G - damping
    
    # Solve for accelerations
    ddq = np.linalg.solve(M, rhs)
    
    return np.concatenate(([dq1, dq2], ddq))

def plant(x_k, u_k, dt=0.01, process_noise_std=1e-4):
    """
    Euler integration step.
    """
    x_dot = plant_dynamics(x_k, u_k)
    x_kp1 = x_k + dt * x_dot
    
    # Add small process noise
    noise = np.random.laplace(4) * process_noise_std
    return x_kp1 + noise

# ============================================================
# 2) RHONN structure (Updated for Control Inputs)
# ============================================================
def sigmoidal(z, beta=0.5):
    """Sigmoid S(z). Beta reduced to widen the active range."""
    z = np.clip(z, -50, 50) 
    return 1.0 / (1.0 + np.exp(-beta * z))

def construct_z_vector(x_est, u_input):
    """
    Features for 2-DOF Arm.
    x_est = [q1, q2, dq1, dq2]
    u_input = [tau1, tau2]
    
    CRITICAL: The acceleration depends Linearly on Tau. 
    So Tau should be passed directly (or scaled), not inside a sigmoid.
    """
    q1, q2, dq1, dq2 = x_est
    tau1, tau2 = u_input
    
    # Scale inputs for sigmoids to avoid saturation
    s_q1 = sigmoidal(q1)
    s_q2 = sigmoidal(q2)
    s_dq1 = sigmoidal(dq1)
    s_dq2 = sigmoidal(dq2)
    
    # Interaction terms (physics based intuition)
    # Cosine terms are common in robotics, approximated by sigmoid interactions
    
    return np.array([
        s_q1, s_q2, s_dq1, s_dq2,     # Individual states
        s_q2 * s_dq1,                 # Coriolis interaction proxy
        s_q2 * s_dq2,                 # Coriolis interaction proxy
        s_dq1**2, s_dq2**2,           # Centrifugal proxies
        tau1 * 0.1, tau2 * 0.1,       # Linear Inputs (Scaled)
        1.0                           # Bias
    ])

# ============================================================
# 3) Trainers (EKF, UKF, PF) - Generic Classes
# ============================================================
# (Reusing the robust classes from previous steps, slightly adapted for input u)

class Generic_RHONN_Trainer:
    """ Base class to handle the loop logic easily """
    def get_prediction(self, weights, x_k, u_k):
        z = construct_z_vector(x_k, u_k)
        return np.dot(weights, z)

class EKF_Trainer(Generic_RHONN_Trainer):
    def __init__(self, n_neurons, n_weights, eta=1.0, P0=1.0, Q=1e-4, R=1e-2):
        self.n_neurons = n_neurons
        self.weights = [np.random.randn(n_weights)*0.1 for _ in range(n_neurons)]
        self.P = [np.eye(n_weights)*P0 for _ in range(n_neurons)]
        self.Q = np.eye(n_weights)*Q
        self.R = R
        self.eta = eta

    def update(self, x_kp1, x_k, u_k):
        z = construct_z_vector(x_k, u_k)
        H = z.reshape(-1, 1)
        
        for i in range(self.n_neurons):
            # Predict P
            P_pred = self.P[i] + self.Q
            
            # Kalman Gain
            S = self.R + (H.T @ P_pred @ H)[0,0]
            K = (P_pred @ H).flatten() / S
            
            # Error
            y_pred = np.dot(self.weights[i], z)
            err = x_kp1[i] - y_pred
            
            # Update Weights
            self.weights[i] += self.eta * K * err
            
            # Update Covariance (Joseph form)
            I_KH = np.eye(len(z)) - np.outer(K, z)
            self.P[i] = I_KH @ P_pred @ I_KH.T + np.outer(K, K)*self.R

class UKF_Trainer(Generic_RHONN_Trainer):
    def __init__(self, n_neurons, n_weights, eta=1.0, alpha=1e-2):
        self.n_neurons = n_neurons
        self.weights = [np.random.randn(n_weights)*0.1 for _ in range(n_neurons)]
        self.P = [np.eye(n_weights) for _ in range(n_neurons)]
        self.Q = np.eye(n_weights)*1e-4
        self.R = 1e-2
        self.eta = eta
        
        # Sigma params
        self.n = n_weights
        self.lambda_ = alpha**2 * (self.n) - self.n
        self.Wm = np.full(2*self.n+1, 1/(2*(self.n+self.lambda_)))
        self.Wc = np.copy(self.Wm)
        self.Wm[0] = self.lambda_/(self.n+self.lambda_)
        self.Wc[0] = self.Wm[0] + (3 - alpha**2)

    def update(self, x_kp1, x_k, u_k):
        z = construct_z_vector(x_k, u_k)
        
        for i in range(self.n_neurons):
            # Generate Sigmas
            try:
                L = np.linalg.cholesky((self.n + self.lambda_) * self.P[i])
            except:
                L = np.eye(self.n) * 0.1 # Fallback
                
            sigmas = np.zeros((2*self.n+1, self.n))
            sigmas[0] = self.weights[i]
            for k in range(self.n):
                sigmas[k+1] = self.weights[i] + L[:,k]
                sigmas[self.n+k+1] = self.weights[i] - L[:,k]
            
            # Transform (Measurement is linear: w^T * z)
            Y_sigmas = np.dot(sigmas, z)
            y_mean = np.sum(self.Wm * Y_sigmas)
            
            # Covariances
            Py = np.sum(self.Wc * (Y_sigmas - y_mean)**2) + self.R
            Pxy = np.zeros(self.n)
            for k in range(2*self.n+1):
                Pxy += self.Wc[k] * (sigmas[k] - self.weights[i]) * (Y_sigmas[k] - y_mean)
                
            # Update
            K = Pxy / Py
            err = x_kp1[i] - y_mean
            self.weights[i] += self.eta * K * err
            self.P[i] -= np.outer(K, K) * Py
            
            # Regularize P
            self.P[i] += np.eye(self.n)*1e-6

class PF_Trainer(Generic_RHONN_Trainer):
    def __init__(self, n_neurons, n_weights, n_particles=100):
        self.n_neurons = n_neurons
        self.n_weights = n_weights
        self.n_particles = n_particles
        self.particles = [np.random.randn(n_particles, n_weights)*0.2 for _ in range(n_neurons)]
        self.weights_pf = [np.ones(n_particles)/n_particles for _ in range(n_neurons)]
        self.Q_std = 0.2
        self.R_std = 0.1

    def update(self, x_kp1, x_k, u_k):
        z = construct_z_vector(x_k, u_k)
        
        for i in range(self.n_neurons):
            # 1. Drift
            self.particles[i] += np.random.randn(self.n_particles, self.n_weights) * self.Q_std
            
            # 2. Weight
            preds = self.particles[i] @ z
            err = x_kp1[i] - preds
            likelihood = np.exp(-0.5 * (err/self.R_std)**2)
            self.weights_pf[i] *= (likelihood + 1e-300)
            self.weights_pf[i] /= np.sum(self.weights_pf[i])
            
            # 3. Resample (Simple systematic)
            eff_N = 1.0 / np.sum(self.weights_pf[i]**2)
            if eff_N < self.n_particles/2:
                indices = np.random.choice(self.n_particles, self.n_particles, p=self.weights_pf[i])
                self.particles[i] = self.particles[i][indices]
                self.weights_pf[i].fill(1.0/self.n_particles)
                
    def get_estimates(self):
        return [np.average(self.particles[i], axis=0, weights=self.weights_pf[i]) for i in range(self.n_neurons)]


# ============================================================
# 4) Simulation Main Loop
# ============================================================
if __name__ == "__main__":
    n_steps = 1000
    dt = 0.01
    t = np.linspace(0, (n_steps-1)*dt, n_steps)
    
    # 4 Neurons for 4 States [q1, q2, dq1, dq2]
    # In reality, we often only identify velocity/accel, but let's do full state map
    n_states = 4
    n_features = 11 # From construct_z_vector size
    
    # Init Trainers
    ekf = EKF_Trainer(n_states, n_features, eta=0.9)
    ukf = UKF_Trainer(n_states, n_features, eta=1.0)
    pf = PF_Trainer(n_states, n_features, n_particles=700)
    
    # Arrays
    x_true = np.zeros((n_steps, 4))
    x_est_ekf = np.zeros((n_steps, 4))
    x_est_ukf = np.zeros((n_steps, 4))
    x_est_pf = np.zeros((n_steps, 4))
    
    # Initial Conditions (Arm hanging down)
    x_true[0] = [-np.pi/2, 0, 0, 0] 
    x_est_ekf[0] = x_true[0]
    x_est_ukf[0] = x_true[0]
    x_est_pf[0]  = x_true[0]
    
    # Excitation Input (Sine waves to move the arm)
    u_hist = np.zeros((n_steps, 2))
    for k in range(n_steps):
        # Time-varying torque to excite dynamics
        tau1 = 30.0 * np.sin(2.0 * t[k]) 
        tau2 = 15.0 * np.cos(3.0 * t[k])
        u_hist[k] = [tau1, tau2]

    print("Simulating 2-DOF Manipulator...")
    
    for k in range(n_steps - 1):
        # 1. Physics Step
        x_true[k+1] = plant(x_true[k], u_hist[k], dt)
        
        # 2. Identify
        # Note: Series-Parallel Architecture uses TRUE x_k and u_k to predict x_kp1
        
        # EKF
        ekf.update(x_true[k+1], x_true[k], u_hist[k])
        z = construct_z_vector(x_true[k], u_hist[k])
        for i in range(4): x_est_ekf[k+1, i] = np.dot(ekf.weights[i], z)
            
        # UKF
        ukf.update(x_true[k+1], x_true[k], u_hist[k])
        for i in range(4): x_est_ukf[k+1, i] = np.dot(ukf.weights[i], z)
            
        # PF
        pf.update(x_true[k+1], x_true[k], u_hist[k])
        w_pf = pf.get_estimates()
        for i in range(4): x_est_pf[k+1, i] = np.dot(w_pf[i], z)
            
        if k % 100 == 0: print(f"Step {k}")

    # ============================================================
    # 5) Visualización - Formato Tesis
    # ============================================================
    
    print("\nGenerando visualizaciones...")
    
    # Configuración de formato para tesis
    thesis_config = {
        'font_family': 'Computer Modern, serif',
        'font_size': 14,
        'title_font_size': 16,
        'legend_font_size': 12,
        'line_width_true': 2.5,
        'line_width_est': 2.0,
        'plot_width': 1000,
        'plot_height': 500,
        'grid_color': 'rgba(200, 200, 200, 0.3)',
        'grid_width': 0.5
    }
    
    # --- Calcular MSE ---
    mse_q1_ekf = np.mean((x_true[:, 0] - x_est_ekf[:, 0])**2)
    mse_q2_ekf = np.mean((x_true[:, 1] - x_est_ekf[:, 1])**2)
    mse_dq1_ekf = np.mean((x_true[:, 2] - x_est_ekf[:, 2])**2)
    mse_dq2_ekf = np.mean((x_true[:, 3] - x_est_ekf[:, 3])**2)
    
    mse_q1_ukf = np.mean((x_true[:, 0] - x_est_ukf[:, 0])**2)
    mse_q2_ukf = np.mean((x_true[:, 1] - x_est_ukf[:, 1])**2)
    mse_dq1_ukf = np.mean((x_true[:, 2] - x_est_ukf[:, 2])**2)
    mse_dq2_ukf = np.mean((x_true[:, 3] - x_est_ukf[:, 3])**2)
    
    mse_q1_pf = np.mean((x_true[:, 0] - x_est_pf[:, 0])**2)
    mse_q2_pf = np.mean((x_true[:, 1] - x_est_pf[:, 1])**2)
    mse_dq1_pf = np.mean((x_true[:, 2] - x_est_pf[:, 2])**2)
    mse_dq2_pf = np.mean((x_true[:, 3] - x_est_pf[:, 3])**2)
    
    mse_total_ekf = mse_q1_ekf + mse_q2_ekf + mse_dq1_ekf + mse_dq2_ekf
    mse_total_ukf = mse_q1_ukf + mse_q2_ukf + mse_dq1_ukf + mse_dq2_ukf
    mse_total_pf = mse_q1_pf + mse_q2_pf + mse_dq1_pf + mse_dq2_pf
    
    # Reporte
    print("\n" + "="*70)
    print("🏆 MEJOR FILTRO: ", end="")
    mse_dict = {'EKF': mse_total_ekf, 'UKF': mse_total_ukf, 'PF': mse_total_pf}
    best_filter = min(mse_dict, key=mse_dict.get)
    print(f"{best_filter} (MSE total: {mse_dict[best_filter]:.6f})")
    print("="*70)
    
    print("\n--- Comparación de Desempeño (MSE) - Manipulador 2-DOF ---")
    print(f"EKF MSE q₁:   {mse_q1_ekf:.6f} | MSE dq₁:   {mse_dq1_ekf:.6f}")
    print(f"EKF MSE q₂:   {mse_q2_ekf:.6f} | MSE dq₂:   {mse_dq2_ekf:.6f}")
    print(f"UKF MSE q₁:   {mse_q1_ukf:.6f} | MSE dq₁:   {mse_dq1_ukf:.6f}")
    print(f"UKF MSE q₂:   {mse_q2_ukf:.6f} | MSE dq₂:   {mse_dq2_ukf:.6f}")
    print(f"PF  MSE q₁:   {mse_q1_pf:.6f} | MSE dq₁:   {mse_dq1_pf:.6f}")
    print(f"PF  MSE q₂:   {mse_q2_pf:.6f} | MSE dq₂:   {mse_dq2_pf:.6f}")
    
    # --- Gráficas por Estado ---
    states_info = [
        {'idx': 0, 'var': 'q₁', 'desc': 'Ángulo Articulación 1', 'y_label': 'Ángulo q₁ (rad)'},
        {'idx': 1, 'var': 'q₂', 'desc': 'Ángulo Articulación 2', 'y_label': 'Ángulo q₂ (rad)'},
        {'idx': 2, 'var': 'dq₁', 'desc': 'Velocidad Articulación 1', 'y_label': 'Velocidad dq₁ (rad/s)'},
        {'idx': 3, 'var': 'dq₂', 'desc': 'Velocidad Articulación 2', 'y_label': 'Velocidad dq₂ (rad/s)'}
    ]
    
    for state_info in states_info:
        i = state_info['idx']
        
        fig = go.Figure()
        
        # --- Estado real (línea negra gruesa) ---
        fig.add_trace(go.Scatter(
            x=t, y=x_true[:, i],
            mode='lines',
            name='Estado Real',
            line=dict(color='#000000', width=thesis_config['line_width_true']),
            showlegend=True
        ))
        
        # --- Estimaciones ---
        fig.add_trace(go.Scatter(
            x=t, y=x_est_ekf[:, i],
            mode='lines',
            name='EKF-RHONN',
            line=dict(color='#1f77b4', width=thesis_config['line_width_est'], dash='dash'),
            showlegend=True
        ))
        
        fig.add_trace(go.Scatter(
            x=t, y=x_est_ukf[:, i],
            mode='lines',
            name='UKF-RHONN',
            line=dict(color='#2ca02c', width=thesis_config['line_width_est'], dash='dot'),
            showlegend=True
        ))
        
        fig.add_trace(go.Scatter(
            x=t, y=x_est_pf[:, i],
            mode='lines',
            name='PF-RHONN',
            line=dict(color='#d62728', width=thesis_config['line_width_est'], dash='dashdot'),
            showlegend=True
        ))
        
        fig.update_layout(
            title={
                'text': f'Estado {state_info["var"]}: {state_info["desc"]} - Manipulador 2-DOF',
                'x': 0.5,
                'xanchor': 'center',
                'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
            },
            xaxis_title='Tiempo (s)',
            yaxis_title=state_info['y_label'],
            xaxis=dict(
                showgrid=True,
                gridcolor=thesis_config['grid_color'],
                gridwidth=thesis_config['grid_width'],
                showline=True,
                linewidth=1.5,
                linecolor='black',
                mirror=True,
                ticks='outside',
                tickwidth=1.5,
                ticklen=5
            ),
            yaxis=dict(
                showgrid=True,
                gridcolor=thesis_config['grid_color'],
                gridwidth=thesis_config['grid_width'],
                showline=True,
                linewidth=1.5,
                linecolor='black',
                mirror=True,
                ticks='outside',
                tickwidth=1.5,
                ticklen=5
            ),
            legend=dict(
                x=0.02,
                y=0.98,
                xanchor='left',
                yanchor='top',
                bgcolor='rgba(255, 255, 255, 0.9)',
                bordercolor='black',
                borderwidth=1,
                font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
            ),
            font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
            plot_bgcolor='white',
            paper_bgcolor='white',
            width=thesis_config['plot_width'],
            height=thesis_config['plot_height'],
            margin=dict(l=80, r=40, t=80, b=60)
        )
        
        fig.show()
    
    # --- Gráfica de errores combinada ---
    fig_err = go.Figure()
    
    errors_info = [
        {'idx': 0, 'label': 'q₁', 'mse_ekf': mse_q1_ekf, 'mse_ukf': mse_q1_ukf, 'mse_pf': mse_q1_pf},
        {'idx': 1, 'label': 'q₂', 'mse_ekf': mse_q2_ekf, 'mse_ukf': mse_q2_ukf, 'mse_pf': mse_q2_pf}
    ]
    
    for err_info in errors_info:
        i = err_info['idx']
        label = err_info['label']
        
        error_ekf = x_true[:, i] - x_est_ekf[:, i]
        error_ukf = x_true[:, i] - x_est_ukf[:, i]
        error_pf = x_true[:, i] - x_est_pf[:, i]
        
        fig_err.add_trace(go.Scatter(
            x=t, y=error_ekf,
            mode='lines',
            name=f'EKF Error {label} (MSE={err_info["mse_ekf"]:.2e})',
            line=dict(color='#1f77b4', width=1.5),
            opacity=0.8
        ))
        
        fig_err.add_trace(go.Scatter(
            x=t, y=error_ukf,
            mode='lines',
            name=f'UKF Error {label} (MSE={err_info["mse_ukf"]:.2e})',
            line=dict(color='#2ca02c', width=1.5, dash='dot'),
            opacity=0.8
        ))
        
        fig_err.add_trace(go.Scatter(
            x=t, y=error_pf,
            mode='lines',
            name=f'PF Error {label} (MSE={err_info["mse_pf"]:.2e})',
            line=dict(color='#d62728', width=1.5, dash='dashdot'),
            opacity=0.8
        ))
    
    fig_err.update_layout(
        title={
            'text': 'Errores de Estimación de Ángulos - Manipulador 2-DOF',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
        },
        xaxis_title='Tiempo (s)',
        yaxis_title='Error de Estimación (rad)',
        xaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        yaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        legend=dict(
            x=0.02,
            y=0.98,
            xanchor='left',
            yanchor='top',
            bgcolor='rgba(255, 255, 255, 0.9)',
            bordercolor='black',
            borderwidth=1,
            font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
        ),
        font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
        plot_bgcolor='white',
        paper_bgcolor='white',
        width=thesis_config['plot_width'],
        height=thesis_config['plot_height'],
        margin=dict(l=80, r=40, t=80, b=60)
    )
    
    fig_err.show()
    
    # --- Gráfica de barras comparando MSE ---
    fig_mse = go.Figure()
    
    filters = ['EKF-RHONN', 'UKF-RHONN', 'PF-RHONN']
    
    fig_mse.add_trace(go.Bar(
        name='Ángulo q₁',
        x=filters,
        y=[mse_q1_ekf, mse_q1_ukf, mse_q1_pf],
        marker_color='#636EFA',
        text=[f'{mse_q1_ekf:.2e}', f'{mse_q1_ukf:.2e}', f'{mse_q1_pf:.2e}'],
        textposition='outside'
    ))
    
    fig_mse.add_trace(go.Bar(
        name='Ángulo q₂',
        x=filters,
        y=[mse_q2_ekf, mse_q2_ukf, mse_q2_pf],
        marker_color='#EF553B',
        text=[f'{mse_q2_ekf:.2e}', f'{mse_q2_ukf:.2e}', f'{mse_q2_pf:.2e}'],
        textposition='outside'
    ))
    
    fig_mse.add_trace(go.Bar(
        name='Velocidad dq₁',
        x=filters,
        y=[mse_dq1_ekf, mse_dq1_ukf, mse_dq1_pf],
        marker_color='#00CC96',
        text=[f'{mse_dq1_ekf:.2e}', f'{mse_dq1_ukf:.2e}', f'{mse_dq1_pf:.2e}'],
        textposition='outside'
    ))
    
    fig_mse.add_trace(go.Bar(
        name='Velocidad dq₂',
        x=filters,
        y=[mse_dq2_ekf, mse_dq2_ukf, mse_dq2_pf],
        marker_color='#AB63FA',
        text=[f'{mse_dq2_ekf:.2e}', f'{mse_dq2_ukf:.2e}', f'{mse_dq2_pf:.2e}'],
        textposition='outside'
    ))
    
    fig_mse.update_layout(
        title={
            'text': 'Comparación de Error Cuadrático Medio (MSE) - Manipulador 2-DOF',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
        },
        xaxis_title='Tipo de Filtro',
        yaxis_title='Error Cuadrático Medio (MSE)',
        yaxis=dict(
            type='log',
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        xaxis=dict(
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        legend=dict(
            x=0.02,
            y=0.98,
            xanchor='left',
            yanchor='top',
            bgcolor='rgba(255, 255, 255, 0.9)',
            bordercolor='black',
            borderwidth=1,
            font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
        ),
        barmode='group',
        font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
        plot_bgcolor='white',
        paper_bgcolor='white',
        width=thesis_config['plot_width'],
        height=thesis_config['plot_height'],
        margin=dict(l=80, r=40, t=100, b=60)
    )
    
    fig_mse.show()
    
    # --- Animación del Robot (Cinemática) ---
    print("\nGenerando animación del manipulador...")
    step = 10  # Submuestreo para velocidad
    
    l1, l2 = 1.0, 1.0
    
    # Posiciones del robot real
    x1 = l1 * np.cos(x_true[::step, 0])
    y1 = l1 * np.sin(x_true[::step, 0])
    x2 = x1 + l2 * np.cos(x_true[::step, 0] + x_true[::step, 1])
    y2 = y1 + l2 * np.sin(x_true[::step, 0] + x_true[::step, 1])
    
    # Posiciones del robot estimado (EKF)
    x1_ekf = l1 * np.cos(x_est_ekf[::step, 0])
    y1_ekf = l1 * np.sin(x_est_ekf[::step, 0])
    x2_ekf = x1_ekf + l2 * np.cos(x_est_ekf[::step, 0] + x_est_ekf[::step, 1])
    y2_ekf = y1_ekf + l2 * np.sin(x_est_ekf[::step, 0] + x_est_ekf[::step, 1])
    
    # Posiciones del robot estimado (UKF)
    x1_ukf = l1 * np.cos(x_est_ukf[::step, 0])
    y1_ukf = l1 * np.sin(x_est_ukf[::step, 0])
    x2_ukf = x1_ukf + l2 * np.cos(x_est_ukf[::step, 0] + x_est_ukf[::step, 1])
    y2_ukf = y1_ukf + l2 * np.sin(x_est_ukf[::step, 0] + x_est_ukf[::step, 1])
    
    # Posiciones del robot estimado (PF)
    x1_pf = l1 * np.cos(x_est_pf[::step, 0])
    y1_pf = l1 * np.sin(x_est_pf[::step, 0])
    x2_pf = x1_pf + l2 * np.cos(x_est_pf[::step, 0] + x_est_pf[::step, 1])
    y2_pf = y1_pf + l2 * np.sin(x_est_pf[::step, 0] + x_est_pf[::step, 1])

    frames = []
    for k in range(len(x1)):
        frames.append(go.Frame(
            data=[
                # Robot real
                go.Scatter(
                    x=[0, x1[k], x2[k]], 
                    y=[0, y1[k], y2[k]], 
                    mode='lines+markers', 
                    line=dict(color='black', width=5), 
                    marker=dict(size=14, color='black'), 
                    name='Robot Real'
                ),
                # Robot estimado (EKF)
                go.Scatter(
                    x=[0, x1_ekf[k], x2_ekf[k]], 
                    y=[0, y1_ekf[k], y2_ekf[k]], 
                    mode='lines+markers', 
                    line=dict(color='#1f77b4', width=3, dash='dash'), 
                    marker=dict(size=10, color='#1f77b4'), 
                    name='Estimación EKF'
                ),
                # Robot estimado (UKF)
                go.Scatter(
                    x=[0, x1_ukf[k], x2_ukf[k]], 
                    y=[0, y1_ukf[k], y2_ukf[k]], 
                    mode='lines+markers', 
                    line=dict(color='#2ca02c', width=3, dash='dot'), 
                    marker=dict(size=10, color='#2ca02c'), 
                    name='Estimación UKF'
                ),
                # Robot estimado (PF)
                go.Scatter(
                    x=[0, x1_pf[k], x2_pf[k]], 
                    y=[0, y1_pf[k], y2_pf[k]], 
                    mode='lines+markers', 
                    line=dict(color='#d62728', width=3, dash='dashdot'), 
                    marker=dict(size=10, color='#d62728'), 
                    name='Estimación PF'
                ),
                # Trayectoria del efector final (real)
                go.Scatter(
                    x=x2[:k+1], 
                    y=y2[:k+1], 
                    mode='lines',
                    line=dict(color='rgba(0,0,0,0.3)', width=2),
                    name='Trayectoria Real',
                    showlegend=False
                ),
                # Trayectoria del efector final (EKF)
                go.Scatter(
                    x=x2_ekf[:k+1], 
                    y=y2_ekf[:k+1], 
                    mode='lines',
                    line=dict(color='rgba(31,119,180,0.3)', width=1.5),
                    name='Trayectoria EKF',
                    showlegend=False
                ),
                # Trayectoria del efector final (UKF)
                go.Scatter(
                    x=x2_ukf[:k+1], 
                    y=y2_ukf[:k+1], 
                    mode='lines',
                    line=dict(color='rgba(44,160,44,0.3)', width=1.5),
                    name='Trayectoria UKF',
                    showlegend=False
                ),
                # Trayectoria del efector final (PF)
                go.Scatter(
                    x=x2_pf[:k+1], 
                    y=y2_pf[:k+1], 
                    mode='lines',
                    line=dict(color='rgba(214,39,40,0.3)', width=1.5),
                    name='Trayectoria PF',
                    showlegend=False
                )
            ],
            name=str(k)
        ))

    fig_anim = go.Figure(
        data=[
            go.Scatter(
                x=[0, x1[0], x2[0]], 
                y=[0, y1[0], y2[0]], 
                mode='lines+markers', 
                line=dict(color='black', width=5),
                marker=dict(size=14, color='black'),
                name='Robot Real'
            ),
            go.Scatter(
                x=[0, x1_ekf[0], x2_ekf[0]], 
                y=[0, y1_ekf[0], y2_ekf[0]], 
                mode='lines+markers', 
                line=dict(color='#1f77b4', width=3, dash='dash'),
                marker=dict(size=10, color='#1f77b4'),
                name='Estimación EKF'
            ),
            go.Scatter(
                x=[0, x1_ukf[0], x2_ukf[0]], 
                y=[0, y1_ukf[0], y2_ukf[0]], 
                mode='lines+markers', 
                line=dict(color='#2ca02c', width=3, dash='dot'),
                marker=dict(size=10, color='#2ca02c'),
                name='Estimación UKF'
            ),
            go.Scatter(
                x=[0, x1_pf[0], x2_pf[0]], 
                y=[0, y1_pf[0], y2_pf[0]], 
                mode='lines+markers', 
                line=dict(color='#d62728', width=3, dash='dashdot'),
                marker=dict(size=10, color='#d62728'),
                name='Estimación PF'
            ),
            go.Scatter(x=[x2[0]], y=[y2[0]], mode='lines', line=dict(color='rgba(0,0,0,0.3)', width=2), showlegend=False),
            go.Scatter(x=[x2_ekf[0]], y=[y2_ekf[0]], mode='lines', line=dict(color='rgba(31,119,180,0.3)', width=1.5), showlegend=False),
            go.Scatter(x=[x2_ukf[0]], y=[y2_ukf[0]], mode='lines', line=dict(color='rgba(44,160,44,0.3)', width=1.5), showlegend=False),
            go.Scatter(x=[x2_pf[0]], y=[y2_pf[0]], mode='lines', line=dict(color='rgba(214,39,40,0.3)', width=1.5), showlegend=False)
        ],
        layout=go.Layout(
            title={
                'text': 'Animación del Manipulador 2-DOF (Real vs Estimación)',
                'x': 0.5,
                'xanchor': 'center',
                'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
            },
            xaxis=dict(
                range=[-2.2, 2.2], 
                scaleanchor="y", 
                scaleratio=1,
                title='x (m)',
                showgrid=True,
                gridcolor=thesis_config['grid_color'],
                showline=True,
                linewidth=1.5,
                linecolor='black',
                mirror=True
            ),
            yaxis=dict(
                range=[-2.2, 2.2],
                title='y (m)',
                showgrid=True,
                gridcolor=thesis_config['grid_color'],
                showline=True,
                linewidth=1.5,
                linecolor='black',
                mirror=True
            ),
            updatemenus=[dict(
                type="buttons", 
                buttons=[
                    dict(label="▶ Reproducir", method="animate", args=[None, {"frame": {"duration": 50, "redraw": True}, "fromcurrent": True}]),
                    dict(label="⏸ Pausar", method="animate", args=[[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}])
                ],
                x=0.1,
                y=0
            )],
            font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
            plot_bgcolor='white',
            paper_bgcolor='white',
            width=thesis_config['plot_width'],
            height=thesis_config['plot_width'],  # Aspecto cuadrado
            legend=dict(
                x=0.02,
                y=0.98,
                xanchor='left',
                yanchor='top',
                bgcolor='rgba(255, 255, 255, 0.9)',
                bordercolor='black',
                borderwidth=1
            )
        ),
        frames=frames
    )
    
    fig_anim.show()
    
    print("\n✅ Visualización completa.")

Simulating 2-DOF Manipulator...
Step 0
Step 100
Step 200
Step 300
Step 400
Step 500
Step 600
Step 700
Step 800
Step 900

Generando visualizaciones...

🏆 MEJOR FILTRO: PF (MSE total: 0.022156)

--- Comparación de Desempeño (MSE) - Manipulador 2-DOF ---
EKF MSE q₁:   0.021544 | MSE dq₁:   0.217126
EKF MSE q₂:   0.014983 | MSE dq₂:   1.119628
UKF MSE q₁:   0.942545 | MSE dq₁:   1.336297
UKF MSE q₂:   0.239742 | MSE dq₂:   6.313597
PF  MSE q₁:   0.000078 | MSE dq₁:   0.000120
PF  MSE q₂:   0.000056 | MSE dq₂:   0.021902



Generando animación del manipulador...



✅ Visualización completa.
